# LTE Rotational Diagram Analysis

This notebook demonstrates Milestone 4: building an LTE rotational diagram from a synthetic multi-transition dataset for one molecule, then estimating the rotational temperature under optically thin LTE assumptions.

## Same-species requirement

A rotational diagram must use multiple transitions from the same molecular species and, ideally, the same emitting region. It should not mix unrelated molecules such as CO, HCN, and HCO+. This demo uses a clearly labelled synthetic molecule, `DemoMol`, with six synthetic transitions generated from a known LTE temperature near 38 K.

## Assumptions and limitations

Assumptions used here:

- LTE excitation.
- Optically thin emission.
- A single rotational temperature component.
- Beam filling factor approximately one.
- Same molecular species and same emitting region.

Limitations in real observations:

- Optical depth can make lines appear too weak for their column density.
- Beam dilution can differ between transitions observed at different frequencies.
- Line blending can bias integrated intensities.
- Calibration uncertainty can dominate over statistical fitting errors.
- Non-LTE excitation can make a straight-line rotational diagram misleading.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from molecular_conditions.rotational_diagram import (
    build_population_diagram,
    fit_rotational_temperature,
)

DATA_DIR = PROJECT_ROOT / "data" / "demo"
FIGURE_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRUE_SYNTHETIC_TROT_K = 38.0

## Load synthetic multi-transition measurements

The table contains integrated intensities in K km/s. These values were generated from a known synthetic LTE model and small perturbations. They are not real molecular database values or telescope measurements.

In [ ]:
line_table_path = DATA_DIR / "demo_rotational_lines.csv"
line_measurements = pd.read_csv(line_table_path, comment="#")
line_measurements

## Build the population diagram

For optically thin emission in brightness-temperature units, the upper-state column density is

`N_u = 8 pi k nu^2 / (h c^3 A_ul) * integral(T_B dv)`.

With upper-state energy in Kelvin, the rotational-diagram relation is

`ln(N_u / g_u) = ln(N_tot / Q(T_rot)) - E_u / T_rot`.

Because we are not using a real partition function for this synthetic molecule, the intercept is reported honestly as `ln(N_tot / Q)`, and the column-density scale is reported as `N_tot / Q(T_rot)`.

In [ ]:
population = build_population_diagram(line_measurements)
population.to_csv(RESULTS_DIR / "synthetic_population_diagram.csv", index=False)

population[[
    "species",
    "transition",
    "upper_energy_k",
    "upper_column_density_cm2",
    "ln_Nu_over_gu",
    "ln_Nu_over_gu_error",
]]

## Fit the rotational temperature

The slope of the line is `-1 / T_rot`. A steeper negative slope means a lower rotational temperature.

In [ ]:
fit = fit_rotational_temperature(population)
fit_summary = fit.as_dict()
pd.DataFrame([fit_summary]).to_csv(
    RESULTS_DIR / "synthetic_rotational_diagram_fit_summary.csv",
    index=False,
)

print(f"Recovered T_rot: {fit.excitation_temperature_k:.2f} K")
print(f"Known synthetic input T_rot: {TRUE_SYNTHETIC_TROT_K:.2f} K")
print(
    "Column density over partition function: "
    f"{fit.column_density_over_partition_function_cm2:.3e} cm^-2"
)
print(f"Reduced chi-square: {fit.reduced_chi_square:.3f}")
fit_summary

In [ ]:
x = population["upper_energy_k"].to_numpy()
y = population["ln_Nu_over_gu"].to_numpy()
yerr = population["ln_Nu_over_gu_error"].to_numpy()
x_fit = np.linspace(x.min() - 3.0, x.max() + 3.0, 200)
y_fit = fit.intercept + fit.slope * x_fit

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.errorbar(x, y, yerr=yerr, fmt="o", color="black", ecolor="0.55", capsize=3, label="DemoMol synthetic lines")
ax.plot(x_fit, y_fit, color="tab:red", lw=2.0, label="Weighted LTE fit")

for _, row in population.iterrows():
    ax.annotate(
        row["transition"],
        (row["upper_energy_k"], row["ln_Nu_over_gu"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8,
    )

ax.set_xlabel("Upper-state energy E_u (K)")
ax.set_ylabel("ln(N_u / g_u) [cm^-2]")
ax.set_title("Synthetic DemoMol LTE rotational diagram")
ax.legend(loc="best")
ax.annotate(
    f"T_rot = {fit.excitation_temperature_k:.1f} K\n"
    f"input T = {TRUE_SYNTHETIC_TROT_K:.1f} K\n"
    f"N/Q = {fit.column_density_over_partition_function_cm2:.2e} cm^-2",
    xy=(0.04, 0.05),
    xycoords="axes fraction",
    bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "0.7"},
)

fig.tight_layout()
fig.savefig(FIGURE_DIR / "synthetic_rotational_diagram.png", dpi=150)
plt.show()